In [ ]:
!pip install -q ultralytics==8.4.183 remotezip huggingface_hub pyyaml

In [ ]:
# !pip install -q remotezip huggingface_hub pyyaml   (rode se faltar algum)
import json, shutil, time
from collections import Counter
from pathlib import Path, PurePosixPath

import yaml
from huggingface_hub import hf_hub_download, hf_hub_url
from remotezip import RemoteZip

REPO = "ibrahimhamamci/DENTEX"
OUT = Path("/kaggle/working/dentex_yolo")

# ---- Decisão metodológica em UM lugar só --------------------------------
# chave = category_id_3 original do DENTEX; valor = id da classe no YOLO (None = descartar)
CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 3}
YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion", 3: "deep_caries"}
# Alternativas (basta trocar as duas linhas acima, sem rebaixar nada):
#  (b) sem impactado:       CLASS_MAP = {0: None, 1: 0, 2: 1, 3: 2}
#                           YOLO_NAMES = {0: "caries", 1: "periapical_lesion", 2: "deep_caries"}
#  (c) cáries unificadas:   CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 1}
#                           YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion"}

# Pastas dentro dos ZIPs (atenção: o ZIP de treino tem 3 subconjuntos com nomes de arquivo parecidos)
TRAIN_ZIP, TRAIN_IMG_DIR = "training_data.zip", "training_data/quadrant-enumeration-disease/xrays/"
TRAIN_JSON = "training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"
VAL_ZIP, VAL_IMG_DIR = "validation_data.zip", "validation_data/quadrant_enumeration_disease/xrays/"


def coco_para_yolo(bbox, w_img, h_img):
    """COCO [x_min, y_min, largura, altura] em pixels -> YOLO [cx, cy, w, h] normalizado em [0, 1]."""
    x, y, w, h = bbox
    x0, y0 = max(0.0, x), max(0.0, y)                      # recorta caixas que escapam da imagem
    x1, y1 = min(float(w_img), x + w), min(float(h_img), y + h)
    if x1 <= x0 or y1 <= y0:
        return None
    return ((x0 + x1) / 2 / w_img, (y0 + y1) / 2 / h_img, (x1 - x0) / w_img, (y1 - y0) / h_img)


def converter(coco, zip_name, img_dir, split):
    """Extrai as imagens do split e grava um .txt de rótulos por imagem."""
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    por_imagem = {}
    for a in coco["annotations"]:
        por_imagem.setdefault(a["image_id"], []).append(a)

    url = hf_hub_url(REPO, f"DENTEX/{zip_name}", repo_type="dataset")
    stats = Counter()
    with RemoteZip(url) as z:
        membros = {PurePosixPath(n).name: n for n in z.namelist() if n.startswith(img_dir) and n.endswith(".png")}
        for img in coco["images"]:
            nome = img["file_name"]
            destino = OUT / "images" / split / nome
            if not destino.exists():                        # idempotente: reexecutar não rebaixa
                for tentativa in range(3):
                    try:
                        with z.open(membros[nome]) as src, open(destino, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        break
                    except Exception as e:                  # rede instável: tenta de novo
                        print("retry", nome, e); time.sleep(2)
            linhas = []
            for a in por_imagem.get(img["id"], []):
                cls = CLASS_MAP[a["category_id_3"]]
                if cls is None:
                    stats["descartadas"] += 1
                    continue
                yolo = coco_para_yolo(a["bbox"], img["width"], img["height"])
                if yolo is None:
                    stats["caixas_invalidas"] += 1
                    continue
                linhas.append(f"{cls} " + " ".join(f"{v:.6f}" for v in yolo))
                stats[f"classe_{cls}"] += 1
            (OUT / "labels" / split / (Path(nome).stem + ".txt")).write_text("\n".join(linhas))  # vazio = fundo
            stats["imagens"] += 1
            stats["imagens_sem_caixa"] += (len(linhas) == 0)
    return stats


# ---- Treino ----
train_json = hf_hub_download(REPO, TRAIN_JSON.replace("training_data/", "DENTEX/training_data/", 1), repo_type="dataset") \
    if False else None
with RemoteZip(hf_hub_url(REPO, f"DENTEX/{TRAIN_ZIP}", repo_type="dataset")) as z:
    train_coco = json.loads(z.read(TRAIN_JSON))
val_coco = json.load(open(hf_hub_download(REPO, "DENTEX/validation_triple.json", repo_type="dataset")))

st_train = converter(train_coco, TRAIN_ZIP, TRAIN_IMG_DIR, "train")
st_val = converter(val_coco, VAL_ZIP, VAL_IMG_DIR, "val")
print("treino:", dict(st_train)); print("validação:", dict(st_val))

# ---- data.yaml: é ele que o YOLO lê no treino ----
(OUT / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(OUT), "train": "images/train", "val": "images/val", "names": YOLO_NAMES},
    sort_keys=False, allow_unicode=True))
print((OUT / "data.yaml").read_text())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def contar(split):
    c, vazios, fora = Counter(), 0, 0
    for f in (OUT / "labels" / split).glob("*.txt"):
        linhas = [l for l in f.read_text().splitlines() if l.strip()]
        vazios += (len(linhas) == 0)
        for l in linhas:
            cls, *xywh = l.split()
            c[int(cls)] += 1
            fora += any(not (0 <= float(v) <= 1) for v in xywh)
    n_img = len(list((OUT / "images" / split).glob("*.png")))
    return n_img, len(list((OUT / "labels" / split).glob("*.txt"))), vazios, fora, dict(sorted(c.items()))

for s in ["train", "val"]:
    print(s, "-> (imagens, rótulos, sem caixa, coords fora de [0,1], caixas por classe):", contar(s))
# Esperado (4 classes): treino (705, 705, 27, 0, {0:604, 1:2189, 2:158, 3:578}); val (50, 50, 0, 0, {0:40, 1:101, 2:9, 3:32})

# Conferência visual: desenha as caixas convertidas de uma imagem com várias lesões
cores = {0: "tab:blue", 1: "tab:red", 2: "tab:green", 3: "tab:orange"}
f = max((OUT / "labels" / "train").glob("*.txt"), key=lambda p: len(p.read_text().splitlines()))
im = Image.open(OUT / "images" / "train" / (f.stem + ".png")); W, H = im.size
fig, ax = plt.subplots(figsize=(14, 7)); ax.imshow(im, cmap="gray"); ax.axis("off")
for l in f.read_text().splitlines():
    cls, cx, cy, w, h = int(l.split()[0]), *map(float, l.split()[1:])
    ax.add_patch(plt.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, ec=cores.get(cls, "yellow"), lw=2))
    ax.text((cx - w / 2) * W, (cy - h / 2) * H - 5, YOLO_NAMES[cls], color=cores.get(cls, "yellow"), fontsize=9)
plt.show()

In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np

OUT = Path("/kaggle/working/dentex_yolo")
NOMES = {0: "impacted", 1: "caries", 2: "periapical_lesion", 3: "deep_caries"}
assert OUT.exists(), "Pasta não encontrada: rode o pip install e a Célula 5 de novo"

def imagens_por_classe(split):
    c = Counter()
    for f in (OUT / "labels" / split).glob("*.txt"):
        c.update({int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()})
    return {NOMES[k]: v for k, v in sorted(c.items())}

for s in ["train", "val"]:
    print(s, "imagens com a classe:", imagens_por_classe(s))

lado = []   # menor lado da caixa, em pixels, com a imagem reduzida para 640 de largura
for f in (OUT / "labels" / "train").glob("*.txt"):
    for l in f.read_text().splitlines():
        if l.strip():
            _, cx, cy, w, h = map(float, l.split()); lado.append(min(w, h) * 640)
print("menor lado da caixa @640 px (p5, mediana, p95):", np.percentile(lado, [5, 50, 95]).round(1))

In [ ]:
from ultralytics import YOLO

SEED = 42
DATA = "/kaggle/working/dentex_yolo/data.yaml"

model = YOLO("yolov8n.pt")            # pesos pré-treinados no COCO (transfer learning)
model.train(
    data=DATA,
    imgsz=640,          # resolução padrão do baseline
    epochs=100,
    patience=20,        # para se a validação não melhorar em 20 épocas
    batch=16,
    seed=SEED,
    deterministic=True,
    device=0,           # uma GPU (evita problemas com o modo multi-GPU)
    workers=2,
    project="/kaggle/working/runs",
    name="baseline_yolov8n_640",
    exist_ok=True,
)

In [ ]:
best = YOLO("/kaggle/working/runs/baseline_yolov8n_640/weights/best.pt")
m = best.val(data=DATA, imgsz=640, split="val", device=0, plots=True)

print(f"Geral: P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")
for k, c in enumerate(m.box.ap_class_index):
    print(f"{m.names[c]:>18}: P={m.box.p[k]:.3f}  R={m.box.r[k]:.3f}  AP50={m.box.ap50[k]:.3f}")

In [ ]:
import pandas as pd, shutil
from pathlib import Path

RUN = Path("/kaggle/working/runs/baseline_yolov8n_640")
df = pd.read_csv(RUN / "results.csv"); df.columns = [c.strip() for c in df.columns]
print("épocas treinadas:", len(df))
print("melhor época (mAP50-95):", int(df["metrics/mAP50-95(B)"].idxmax()) + 1, "| mAP50-95 =", round(df["metrics/mAP50-95(B)"].max(), 3))
print(df[["epoch", "train/box_loss", "val/box_loss", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5).round(3))

pasta = Path("/kaggle/working/baseline_artefatos"); pasta.mkdir(exist_ok=True)
for nome in ["results.csv", "results.png", "args.yaml", "confusion_matrix.png",
             "confusion_matrix_normalized.png", "BoxPR_curve.png", "weights/best.pt"]:
    origem = RUN / nome
    if origem.exists():
        shutil.copy(origem, pasta / origem.name)
print(sorted(p.name for p in pasta.iterdir()))

In [ ]:
# Matriz de confusão num limiar realista (0,25), só para a figura.
# As métricas de mAP continuam sendo as da validação com conf=0,001.
best = YOLO("/kaggle/working/runs/baseline_yolov8n_640/weights/best.pt")
m25 = best.val(data=DATA, imgsz=640, split="val", device=0, conf=0.25,
               plots=True, project="/kaggle/working/runs", name="val_conf025", exist_ok=True)
print(f"conf=0,25 -> P={m25.box.mp:.3f}  R={m25.box.mr:.3f}  (mAP aqui não é o oficial)")

import shutil
for n in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
    shutil.copy(f"/kaggle/working/runs/val_conf025/{n}", f"/kaggle/working/baseline_artefatos/{n.replace('.png','_conf025.png')}")

In [ ]:
# Última célula: gera o zip depois de todas as figuras estarem copiadas.
import shutil
shutil.make_archive("/kaggle/working/baseline_artefatos", "zip", "/kaggle/working/baseline_artefatos")
